In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent  # if notebook is in task_3/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import pathlib

from dotenv import load_dotenv

# from config import STORE_DIR
from task_3.indexing.data_chunker import DataChunker, print_chunk_length_stats
from task_3.indexing.resumable_indexer import run_resumable_indexing
from llm_client import LLMClient
from task_3.indexing.vector_store import VectorStore
load_dotenv()

True

In [3]:
from config import EXTRACTED_MEDIA_DIR, STORE_DIR

data_path = pathlib.Path().cwd().parent / "data"
disruption_plans_path = data_path / "disruption_plans"
print(pathlib.Path().cwd())
print(f"Store: {STORE_DIR.resolve()}")

/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/task_3
Store: /Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/.store


In [4]:
vector_store = VectorStore()

In [5]:
print(f"Collection has {vector_store.count()} existing documents")

Collection has 6166 existing documents


In [6]:
llm_client = LLMClient()
chunker = DataChunker(llm_client)

DataChunker ready (store: /Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/.store)
  extracted_media: /Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/.store/extracted_media
  image cache: /Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/.store/image_descriptions_cache.json


In [7]:
print(disruption_plans_path)

/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/data/disruption_plans


In [8]:
all_paths = list(disruption_plans_path.rglob("*"))
# only files not directories

all_paths = [path for path in all_paths if path.is_file()]
# where path is doc, docx or pptx
all_paths = [path for path in all_paths if path.suffix in [".doc", ".docx", ".pptx", ".pptm"]]
# print(len(all_paths))
for path in all_paths:
    print(path)


/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/data/disruption_plans/Mainline CPT (8th May 21).pptm
/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/data/disruption_plans/20 Batch 2 LOR 2-4 CPT (19th May 21).pptm
/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/data/disruption_plans/SWR Station Disruption Plans/RM Central/Station Disruption Plan - Hampton Wick  Issue 1 - April 2018.docx
/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/data/disruption_plans/SWR Station Disruption Plans/RM Central/Station Disruption Plan - Brookwood Issue 1 - April 2018.docx
/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/data/disruption_plans/SWR Station Disruption Plans/RM Central/Station Disruption Plan - Worplesdon Issue 1 - April 2018.docx
/Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/data/disruption_plans/SWR Station Disruption Plans/RM Central/Station Disruption Plan - Tolworth  Issue 1 - April 2018.docx
/Users/bec

In [9]:
# chunk_paths = [
#     "RM Central/Station Disruption Plan - Aldershot Issue 1 - April 2018.docx",
#     "RM Waterloo/Station Disruption Plan Waterloo - Issue 1.doc",
#     "RM West/Station Disruption Plan - Weymouth Issue 1 - April 2018.docx"
# ]

In [10]:
chunk_paths = all_paths
# chunk_paths = all_paths[:30]

In [12]:
checkpoint_path = STORE_DIR / "indexing_checkpoint.json"

In [24]:
checkpoint_path = STORE_DIR / "indexing_checkpoint.json"
summary = run_resumable_indexing(
    paths=chunk_paths,
    chunker=chunker,
    vector_store=vector_store,
    checkpoint_path=checkpoint_path,
    max_workers=1,
    retry_failed=True,
)
print("\n--- Resumable indexing summary ---")
print(summary)
print(f"Collection total: {vector_store.count()} documents")
print(f"Checkpoint: {checkpoint_path}")


No pending files to process (all done or failed with retry_failed=False).

--- Resumable indexing summary ---
{'total_input': 199, 'pending': 0, 'processed': 0, 'succeeded': 0, 'failed': 0, 'chunks_upserted': 0}
Collection total: 6200 documents
Checkpoint: /Users/becca/Coursework/sem2/ai/cw2/Advanced-AI-Coursework-II/.store/indexing_checkpoint.json


In [ ]:
chunked = chunker.chunk_directory(disruption_plans_path)
all_chunks = [chunk for chunks in chunked.values() for chunk in chunks]
n = vector_store.add_chunks(all_chunks)
print(f"\nIndexed {n} chunks across {len(chunked)} files")
print(f"Collection now has {vector_store.count()} documents")